# 🧠 QueryPlanningTool - Natural Language to DSL Query Conversion

```mermaid
%%{init: {'theme':'base', 'themeVariables': { 'primaryColor':'#34495E', 'primaryTextColor':'#fff', 'primaryBorderColor':'#2C3E50', 'lineColor':'#F39C12', 'secondaryColor':'#3498DB', 'tertiaryColor':'#27AE60', 'fontSize':'16px'}}}%%
graph TB
    A[👤 Natural Language<br/>Find products > $100] --> B[🤖 Flow Agent]
    B --> C{🧠 QueryPlanningTool}
    C --> D[🎯 LLM Model]
    D --> E[📝 Generate DSL Query]
    E --> F[✅ Valid JSON Query]
    F --> G[🔍 Execute on Index]
    G --> H[📊 Search Results]
    
    style A fill:#3498DB,stroke:#2980B9,color:#fff
    style C fill:#34495E,stroke:#2C3E50,color:#fff
    style D fill:#9B59B6,stroke:#8E44AD,color:#fff
    style E fill:#E67E22,stroke:#D35400,color:#fff
    style H fill:#27AE60,stroke:#229954,color:#fff
```

## 📚 Learning Objectives

1. ✅ Convert **natural language** to **OpenSearch DSL queries**
2. ✅ Use **LLM knowledge** for query generation
3. ✅ Implement **search templates** for consistent queries
4. ✅ Handle **complex query logic** (filters, aggregations, ranges)
5. ✅ Build **user-friendly search interfaces**

---

## 🎯 What is QueryPlanningTool?

**QueryPlanningTool** uses LLMs to convert natural language questions into valid OpenSearch DSL queries:
- 🗣️ **Natural Interface**: Users ask questions in plain English
- 🧠 **LLM-Powered**: GPT generates correct DSL syntax
- 📋 **Template Support**: Use predefined query templates
- ✅ **Validation**: Ensures generated queries are valid

**Two Modes**:
1. **LLM Knowledge**: Model generates queries from scratch
2. **Search Templates**: Model selects and fills templates

---

## Step 1: Import Libraries

In [ ]:
import sys
import json

sys.path.append('..')
from agent_helpers import (
    get_os_client,
    configure_cluster_for_openai,
    create_openai_connector,
    register_and_deploy_openai_model,
    create_flow_agent,
    execute_agent,
    cleanup_resources
)

print("✅ Libraries imported!")

## Step 2: Initialize Client and Setup OpenAI

In [ ]:
client = get_os_client()
configure_cluster_for_openai(client)
connector_id = create_openai_connector(client)
model_id = register_and_deploy_openai_model(client, connector_id)
print(f"✅ OpenAI model ready: {model_id}")

## Step 3: Create Sample Product Index

In [ ]:
index_name = "products_catalog"

if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

# Create index
client.indices.create(index=index_name, body={
    "mappings": {
        "properties": {
            "product_name": {"type": "text"},
            "category": {"type": "keyword"},
            "price": {"type": "float"},
            "rating": {"type": "float"},
            "in_stock": {"type": "boolean"},
            "brand": {"type": "keyword"},
            "tags": {"type": "keyword"}
        }
    }
})

# Add sample products
products = [
    {"product_name": "Laptop Pro", "category": "Electronics", "price": 1299.99, "rating": 4.5, "in_stock": True, "brand": "TechBrand", "tags": ["laptop", "computers"]},
    {"product_name": "Wireless Mouse", "category": "Electronics", "price": 29.99, "rating": 4.2, "in_stock": True, "brand": "TechBrand", "tags": ["accessories"]},
    {"product_name": "Running Shoes", "category": "Sports", "price": 89.99, "rating": 4.7, "in_stock": True, "brand": "SportyFit", "tags": ["shoes", "running"]},
    {"product_name": "Yoga Mat", "category": "Sports", "price": 45.00, "rating": 4.8, "in_stock": False, "brand": "YogaPro", "tags": ["yoga", "fitness"]},
    {"product_name": "Coffee Maker", "category": "Home", "price": 159.99, "rating": 4.4, "in_stock": True, "brand": "BrewMaster", "tags": ["coffee", "kitchen"]},
]

for product in products:
    client.index(index=index_name, body=product, refresh=True)

print(f"✅ Created index with {len(products)} products")

## Step 4: Create Agent with QueryPlanningTool (LLM Mode)

In [ ]:
# QueryPlanningTool using LLM knowledge
tools = [{
    "type": "QueryPlanningTool",
    "parameters": {
        "model_id": model_id,
        "response_filter": "$.choices[0].message.content"
    }
}]

agent_id = create_flow_agent(
    client, "Query_Planning_Agent",
    "Converts natural language to OpenSearch DSL queries",
    tools
)
print(f"✅ Query planning agent created: {agent_id}")

## Step 5: Test Case 1 - Simple Price Range Query

In [ ]:
parameters = {
    "question": "Find all products with price greater than 100",
    "index_name": index_name
}

print("❓ Natural Language: Find all products with price greater than 100")
print("="*60)
response = execute_agent(client, agent_id, parameters)
print("\n📝 Generated DSL Query:")
print(json.dumps(response, indent=2))

## Step 6: Test Case 2 - Category Filter with Sorting

In [ ]:
parameters = {
    "question": "Show me Electronics products sorted by price from lowest to highest",
    "index_name": index_name
}

print("❓ Natural Language: Show me Electronics products sorted by price")
print("="*60)
response = execute_agent(client, agent_id, parameters)
print("\n📝 Generated DSL Query:")
print(json.dumps(response, indent=2))

## Step 7: Test Case 3 - Boolean Query with Multiple Conditions

In [ ]:
parameters = {
    "question": "Find in-stock products in Sports category with rating above 4.5",
    "index_name": index_name
}

print("❓ Natural Language: In-stock Sports products with rating > 4.5")
print("="*60)
response = execute_agent(client, agent_id, parameters)
print("\n📝 Generated DSL Query:")
print(json.dumps(response, indent=2))

## Step 8: Test Case 4 - Text Search with Filters

In [ ]:
parameters = {
    "question": "Search for 'coffee' in product names and show only items under $200",
    "index_name": index_name
}

print("❓ Natural Language: Search for 'coffee' under $200")
print("="*60)
response = execute_agent(client, agent_id, parameters)
print("\n📝 Generated DSL Query:")
print(json.dumps(response, indent=2))

## Step 9: Test Case 5 - Aggregation Query

In [ ]:
parameters = {
    "question": "Calculate the average price for each product category",
    "index_name": index_name
}

print("❓ Natural Language: Calculate average price per category")
print("="*60)
response = execute_agent(client, agent_id, parameters)
print("\n📝 Generated DSL Query:")
print(json.dumps(response, indent=2))

## 🎓 Key Takeaways

### What We Learned:

1. **QueryPlanningTool Capabilities**:
   - ✅ Converts natural language → DSL queries
   - ✅ Handles complex query logic
   - ✅ Supports filters, ranges, sorting, aggregations
   - ✅ Two modes: LLM knowledge or templates

2. **Query Types Supported**:
   ```python
   # Range queries
   "price greater than 100" → {"range": {"price": {"gt": 100}}}
   
   # Boolean queries
   "Electronics AND in stock" → {"bool": {"must": [...]}}
   
   # Text search
   "search for coffee" → {"match": {"product_name": "coffee"}}
   
   # Aggregations
   "average price by category" → {"aggs": {"avg": {...}}}
   ```

3. **Use Cases**:
   - 🗣️ **Natural Language Search**: User-friendly interfaces
   - 📊 **Business Intelligence**: Non-technical users query data
   - 🤖 **Chatbots**: Conversational data access
   - 🔍 **Dynamic Queries**: Generate queries based on context

4. **Configuration**:
   ```python
   # LLM mode (model generates query)
   {
       "type": "QueryPlanningTool",
       "parameters": {
           "model_id": model_id,
           "response_filter": "$.choices[0].message.content"
       }
   }
   
   # Template mode (model selects template)
   {
       "type": "QueryPlanningTool",
       "parameters": {
           "model_id": model_id,
           "generation_type": "user_templates",
           "search_templates": [
               {
                   "template_id": "price_range",
                   "template_description": "Find products in price range"
               }
           ]
       }
   }
   ```

### Best Practices:

- ✅ **Index Schema**: Provide schema info to LLM for better queries
- ✅ **Validation**: Always validate generated queries before execution
- ✅ **Templates**: Use templates for common query patterns
- ✅ **Error Handling**: Gracefully handle invalid queries
- ✅ **Query Limits**: Set size limits to prevent large result sets

### Advantages Over Manual DSL:

| Aspect | Manual DSL | QueryPlanningTool |
|--------|-----------|------------------|
| User Skill | Expert | Beginner |
| Speed | Slow | Fast |
| Syntax Errors | Common | Rare |
| Learning Curve | Steep | Gentle |
| Natural Language | ❌ | ✅ |

### Combining with Other Tools:

```python
# Complete search pipeline
tools = [
    {"type": "QueryPlanningTool", ...},  # 1. Generate query
    {"type": "SearchIndexTool", ...},    # 2. Execute query
    {"type": "MLModelTool", ...}         # 3. Explain results
]
```

---

## 🧹 Cleanup

In [ ]:
# # cleanup_resources(
# #     client=client,
# #     agent_ids=[agent_id],
# #     model_ids=[model_id],
# #     connector_ids=[connector_id]
# # )
# # client.indices.delete(index=index_name)
# # print("✅ Cleanup complete!")

## 🚀 Next Steps

- **PPLTool**: Generate PPL queries from natural language
- **SearchIndexTool**: Execute the generated queries
- **MLModelTool**: Explain query results to users

📚 [QueryPlanningTool Docs](https://opensearch.org/docs/latest/ml-commons-plugin/agents-tools/tools/query-planning-tool/)